In [1]:
!pip install boto3 awswrangler pandas scikit-learn joblib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.0/374.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.0 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'ap-southeast-2'

print("✅ AWS credentials loaded securely!")

✅ AWS credentials loaded securely!


In [3]:
import awswrangler as wr
import pandas as pd

# Pull data from silver layer
df = wr.athena.read_sql_query(
    """
    SELECT
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        c.customer_state,
        COUNT(oi.order_item_id) as num_items,
        SUM(oi.price) as total_price,
        SUM(oi.freight_value) as total_freight
    FROM olist_silver_db.orders o
    JOIN olist_silver_db.customers c ON o.customer_id = c.customer_id
    JOIN olist_silver_db.order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    AND o.order_delivered_customer_date != ''
    AND o.order_purchase_timestamp != ''
    GROUP BY
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        c.customer_state
    """,
    database="olist_silver_db",
    s3_output="s3://olist-ecommerce-pipeline-john/output/"
)

print(f"✅ Data pulled! Shape: {df.shape}")
df.head()

✅ Data pulled! Shape: (96470, 8)


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,customer_state,num_items,total_price,total_freight
0,96a829e0e554c697670638997aadc45b,delivered,2017-04-19 15:02:03,2017-05-15 13:16:14,SP,1,44.99,10.96
1,1e219f956f9c6addbb50e09227473228,delivered,2017-11-28 18:13:33,2017-12-30 13:13:25,SP,1,78.97,21.21
2,b5685de8123bd5d22f75055f38b6cf7e,delivered,2017-03-18 14:39:09,2017-03-29 12:35:31,GO,1,29.90,17.78
3,51119a98965b7d1f2fee96f39ad86695,delivered,2017-05-02 15:37:00,2017-05-15 13:52:02,PR,2,359.80,87.44
4,003edccf16bc5ec447f592913b3df2b4,delivered,2018-07-07 09:02:51,2018-08-02 23:07:33,MA,1,14.00,50.85


In [4]:
# Check the data
print(df.dtypes)
print("\n")
print(df.head())
print("\n")
print(df.isnull().sum())

order_id                         string[python]
order_status                     string[python]
order_purchase_timestamp         string[python]
order_delivered_customer_date    string[python]
customer_state                   string[python]
num_items                                 Int64
total_price                             float64
total_freight                           float64
dtype: object


                           order_id order_status order_purchase_timestamp  \
0  96a829e0e554c697670638997aadc45b    delivered      2017-04-19 15:02:03   
1  1e219f956f9c6addbb50e09227473228    delivered      2017-11-28 18:13:33   
2  b5685de8123bd5d22f75055f38b6cf7e    delivered      2017-03-18 14:39:09   
3  51119a98965b7d1f2fee96f39ad86695    delivered      2017-05-02 15:37:00   
4  003edccf16bc5ec447f592913b3df2b4    delivered      2018-07-07 09:02:51   

  order_delivered_customer_date customer_state  num_items  total_price  \
0           2017-05-15 13:16:14             SP          1      

In [5]:
import pandas as pd
from datetime import datetime

# Convert timestamps to datetime
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])

# Create target variable — delivery days
df['delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

# Extract useful features from purchase date
df['purchase_month'] = df['order_purchase_timestamp'].dt.month
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour

# Drop rows with negative delivery days (data quality issue)
df = df[df['delivery_days'] > 0]

# Check the result
print(f"✅ Features engineered! Shape: {df.shape}")
print(f"\nDelivery days stats:")
print(df['delivery_days'].describe())

✅ Features engineered! Shape: (96457, 12)

Delivery days stats:
count    96457.000000
mean        12.095234
std          9.550992
min          1.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64


In [6]:
from sklearn.preprocessing import LabelEncoder

# Select features for model
features = [
    'customer_state',
    'num_items',
    'total_price',
    'total_freight',
    'purchase_month',
    'purchase_dayofweek',
    'purchase_hour'
]

target = 'delivery_days'

# Encode customer_state (convert text to numbers)
le = LabelEncoder()
df['customer_state_encoded'] = le.fit_transform(df['customer_state'])

# Update features list
features = [
    'customer_state_encoded',
    'num_items',
    'total_price',
    'total_freight',
    'purchase_month',
    'purchase_dayofweek',
    'purchase_hour'
]

# Remove outliers (delivery days > 60)
df = df[df['delivery_days'] <= 60]

# Create X and y
X = df[features]
y = df[target]

print(f"✅ Features ready!")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeatures: {features}")

✅ Features ready!
X shape: (96169, 7)
y shape: (96169,)

Features: ['customer_state_encoded', 'num_items', 'total_price', 'total_freight', 'purchase_month', 'purchase_dayofweek', 'purchase_hour']


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Train Random Forest model
print("\n⏳ Training model...")
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n✅ Model trained!")
print(f"Mean Absolute Error: {mae:.2f} days")
print(f"R2 Score: {r2:.3f}")

Training set: (76935, 7)
Test set: (19234, 7)

⏳ Training model...

✅ Model trained!
Mean Absolute Error: 4.82 days
R2 Score: 0.329


In [8]:
import pandas as pd

# Feature importance
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(importance)

Feature Importance:
                  feature  importance
0  customer_state_encoded    0.491549
4          purchase_month    0.204296
3           total_freight    0.198430
2             total_price    0.047820
6           purchase_hour    0.024428
1               num_items    0.017619
5      purchase_dayofweek    0.015858


In [9]:
import joblib
import boto3

# Save model locally first
joblib.dump(model, 'delivery_model.pkl')
joblib.dump(le, 'label_encoder.pkl')

# Upload to S3
s3 = boto3.client('s3', region_name='ap-southeast-2')

s3.upload_file(
    'delivery_model.pkl',
    'olist-ecommerce-pipeline-john',
    'models/delivery_model.pkl'
)

s3.upload_file(
    'label_encoder.pkl',
    'olist-ecommerce-pipeline-john',
    'models/label_encoder.pkl'
)

print("✅ Model saved to S3!")
print("s3://olist-ecommerce-pipeline-john/models/delivery_model.pkl")
print("s3://olist-ecommerce-pipeline-john/models/label_encoder.pkl")

✅ Model saved to S3!
s3://olist-ecommerce-pipeline-john/models/delivery_model.pkl
s3://olist-ecommerce-pipeline-john/models/label_encoder.pkl
